<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/m05-clustering-and-topic-modeling/guided-labs/AI301_Guided_Lab_Part2_Library_Topics_From_Clusters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Guided Lab: Clustering and Topic Modeling, Part 2 · Topics From Clusters

**Module 5 · Clustering and Topic Modeling**

> **File > Save a copy in Drive** before you start. Switch the runtime on: **Runtime > Change
> runtime type > T4 GPU**. This notebook downloads one dataset of about 53 MB, one embedding model
> of about 67 MB and one generative model of about 308 MB.

Part 1 of this lab grouped 5,000 abstracts into clusters and read them by hand. This notebook uses
BERTopic to describe each cluster with words, then tries three other ways to describe the same
clusters:

1. **c-TF-IDF**: the keywords BERTopic produces by counting words, Part 2.
2. **KeyBERTInspired**: keywords chosen by embedding similarity, Part 4.
3. **Maximal Marginal Relevance**: keywords chosen to avoid near duplicates, Part 4.
4. **Flan-T5**: a short label written by a generative model, Part 4.

In [ ]:
%pip install -q bertopic

---

## Part 0 · Setup

This is a separate notebook, so the sample, the embeddings and the two models from Part 1 of this
lab have to be built again. The code is the same as Parts 1 to 4 there.

> Dataset card: [maartengr/arxiv_nlp](https://huggingface.co/datasets/maartengr/arxiv_nlp). Model
> card: [thenlper/gte-small](https://huggingface.co/thenlper/gte-small). Both are the same as in
> Part 1 of this lab; read them there if you have not.

In [ ]:
# UMAP, scikit-learn and transformers print warnings and load reports, none of them is an error
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# the same 5,000 abstracts as Part 1
# You can test on the full 40k+ dataset
dataset = load_dataset("maartengr/arxiv_nlp", split="train")
dataset = dataset.shuffle(seed=42).select(range(5000))

abstracts = list(dataset["Abstracts"])
titles = list(dataset["Titles"])

# the same three models as Part 1
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

umap_model = UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=20, metric="euclidean", cluster_selection_method="eom")

print("abstracts:", len(abstracts))
print("embeddings:", embeddings.shape)

---

## Part 1 · BERTopic

BERTopic runs the pipeline from Part 1 of this lab and adds one step at the end:

1. Embed each document.
2. Reduce the embeddings with UMAP.
3. Cluster the reduced embeddings with HDBSCAN.
4. Describe each cluster with the words that set it apart from the other clusters.

Steps 1 to 3 are the same as before, and we pass BERTopic the same models. Step 4 is new, and Part 2
explains it. BERTopic calls each cluster a topic and numbers the topics by size, so topic 0 is the
largest. Topic `-1` holds the outliers.

- **Python note:** `fit(abstracts, embeddings)` gives BERTopic the vectors we already computed, so
  it skips step 1 instead of embedding the abstracts a second time. `get_topic_info()` returns a
  pandas DataFrame with one row per topic.

> **Predict before running.**
>
> BERTopic gets the same embeddings, the same UMAP settings and the same HDBSCAN settings as Part 1.
> Will it find the same number of topics as HDBSCAN found clusters there? Say why or why not.

**Your prediction:**

_Type here._

> Library reference: [BERTopic documentation](https://maartengr.github.io/BERTopic/index.html).
> The algorithm page shows the four steps above and which model each one can be swapped for.

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True,
)
topic_model.fit(abstracts, embeddings)

topic_info = topic_model.get_topic_info()

# count every topic except -1, the outliers
n_topics = 0
for topic in topic_info["Topic"]:
    if topic != -1:
        n_topics = n_topics + 1

print("topics, not counting -1:", n_topics)
topic_info.head(10)

**Reading the table.** Each row is one topic. The unnamed first column is the pandas row number;
the topic number is in `Topic`.

| Column | What it holds |
|---|---|
| `Topic` | the topic number; 0 is the largest topic and `-1` holds the outliers |
| `Count` | how many abstracts are in the topic |
| `Name` | the topic number and its top four keywords in one string, made automatically |
| `Representation` | the top ten c-TF-IDF keywords, BERTopic's description of the topic |
| `Representative_Docs` | three abstracts that best match the topic; read them to check the keywords |

**Interpret the output.**

1. Compare the number of topics and the size of topic `-1` with the clusters and outliers you
   recorded in Part 1. Say whether your prediction held, and what a difference would tell you.
2. Look at the `Name` of topic `-1`. Say why its words look the way they do, given what the
   outliers are.
3. Read the three `Representative_Docs` of topic 0. Say whether they match its keywords.

**Your answer:**

_Type here._

---

## Part 2 · Reading the topic keywords

BERTopic already ran c-TF-IDF inside `fit` in Part 1. The cells below only read the keywords it
stored; this part explains how they were scored.

In AI 102 you used TF-IDF, which gives a word a high score in a document when the word is frequent
in that document and rare across the other documents. c-TF-IDF applies the same idea to clusters:

1. Join all the documents in a cluster into one long document.
2. Count how often each word appears in it.
3. Give a word a high score when it is frequent in this cluster and rare in the other clusters.

The top scoring words become the topic's keywords. They come from counting, not from reading the
abstracts.

`get_topic(0)` returns the keywords of topic 0 as a list of (word, score) pairs, highest score
first. `visualize_barchart` draws the top keywords of several topics side by side.

- **Python note:** `for word, score in ...` unpacks each pair into two variables in the loop header.

> **Predict before running.**
>
> Will a common word such as "the" or "model" appear among the top keywords of any topic?

**Your prediction:**

_Type here._

In [ ]:
# the keywords of the largest topic and their c-TF-IDF scores
for word, score in topic_model.get_topic(0):
    print(word, round(float(score), 4))

In [ ]:
# the top keywords of the eight largest topics
topic_model.visualize_barchart(top_n_topics=8)

**Interpret the output.**

1. Give three of the eight topics a name of two or three words from their keywords alone. Then say
   which one you are least sure of, and what you would read to check it.
2. Find a keyword that says little about its topic, if there is one. Say how c-TF-IDF can let such a
   word through.

**Your answer:**

_Type here._

> **Your turn.**
>
> Call `topic_model.get_topic` on a small topic near the bottom of `topic_info`. Say whether its
> keywords are easier or harder to name than those of topic 0, and why the size of the cluster
> would matter for c-TF-IDF.

In [ ]:
# Your turn: keywords of a small topic.

---

## Part 3 · Searching the topics

`find_topics` takes a search phrase and returns the topics closest to it. BERTopic keeps one vector
per topic, the average of the embeddings of its documents. `find_topics` embeds the phrase with the
same model and compares it with every topic vector by cosine similarity, the measure from Module 3.
This is the class averages method from Part 2 of the Module 4 guided lab, applied to topics instead
of labels.

- **Python note:** `find_topics` returns two lists, the topic numbers and their similarity scores,
  in the same order. `[:5]` keeps the first five words of a topic's keyword list.

> **Predict before running.**
>
> 1. Will the best match for "topic modeling" score clearly higher than the fifth best, or only
>    slightly?
> 2. What will `find_topics` return for a phrase that matches no topic at all?

**Your prediction:**

_Type here._

In [ ]:
search_phrase = "topic modeling"

similar_topics, similarity = topic_model.find_topics(search_phrase, top_n=5)

for i in range(len(similar_topics)):
    topic = similar_topics[i]

    words = []
    for word, score in topic_model.get_topic(topic)[:5]:
        words.append(word)

    print("topic", topic, "| similarity", round(float(similarity[i]), 3), "|", ", ".join(words))

**Interpret the output.**

1. Look at the gap between the first and the fifth similarity. Say what a small gap means for how
   much you can trust the top result.
2. Change `search_phrase` to a subject you expect is not in NLP research, such as a sport, and
   rerun. Say what came back and why `find_topics` always returns something.

**Your answer:**

_Type here._

---

## Part 4 · Representation models

Starting a new session? Run Parts 0 and 1 first, then continue here.

c-TF-IDF keywords are BERTopic's first description of each topic. A representation model replaces
that description. It does not move any document: the topics and their sizes stay the same, and only
the words describing them change.

We compare four descriptions of the same five topics:

| Description | How it is made |
|---|---|
| c-TF-IDF | the keywords from Part 2 |
| KeyBERTInspired | takes the c-TF-IDF keywords as candidates, averages the embeddings of the topic's most representative documents into one topic vector, and keeps the candidates whose embeddings are closest to it by cosine |
| Maximal Marginal Relevance | reorders the c-TF-IDF keywords so that each new keyword is relevant to the topic and unlike the keywords already picked; `diversity` runs from 0, no change, to 1, most varied |
| Flan-T5 | a generative model reads the keywords and three representative abstracts and writes a short label |

`update_topics` recomputes the c-TF-IDF keywords and then applies the representation model, so each
call starts from the same keywords.

The function `top_words` returns the first five keywords of a topic as one string, so the four
descriptions can be stored side by side in a table.

- **Python note:** `", ".join(words)` joins a list of strings into one string with a comma and a
  space between the items.

> **Predict before running.**
>
> 1. Which of the three keyword methods will change the c-TF-IDF keywords the most?
> 2. Will a small generative model write a better label than the keywords, or a vaguer one?

**Your prediction:**

_Type here._

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance


def top_words(model, topic, n=5):
    """Return the first n keywords of one topic as a single string."""
    words = []
    for word, score in model.get_topic(topic)[:n]:
        words.append(word)
    return ", ".join(words)


# one dictionary per topic, filled in one description at a time
rows = []
for topic in range(5):
    rows.append({"topic": topic, "c-TF-IDF": top_words(topic_model, topic)})

# KeyBERTInspired
topic_model.update_topics(abstracts, representation_model=KeyBERTInspired())
for topic in range(5):
    rows[topic]["KeyBERTInspired"] = top_words(topic_model, topic)

# Maximal Marginal Relevance
topic_model.update_topics(abstracts, representation_model=MaximalMarginalRelevance(diversity=0.5))
for topic in range(5):
    rows[topic]["MMR"] = top_words(topic_model, topic)

# back to the plain c-TF-IDF keywords before the next step
topic_model.update_topics(abstracts)

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

Flan-T5 is the encoder decoder model from Part 3 of the Module 4 guided lab. For each topic we
build one prompt from the topic's keywords and its three most representative abstracts, cut to their
first 50 words, and ask for a short label.

- **Python note:** `get_representative_docs(topic)` returns the abstracts BERTopic found most
  typical of the topic, three by default. `split()` breaks a string into words and `" ".join(...)`
  puts the first 50 back together.

> Model card: [google/flan-t5-small](https://huggingface.co/google/flan-t5-small). Read it for the
> size of the model and the tasks it was fine tuned on, since both limit how good a label it can
> write.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", device)

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(device)

for topic in range(5):
    keywords = top_words(topic_model, topic, n=10)

    # the first 50 words of each representative abstract
    documents = ""
    for doc in topic_model.get_representative_docs(topic):
        documents = documents + "- " + " ".join(doc.split()[:50]) + "\n"

    prompt = (
        "I have a topic that contains the following documents:\n"
        + documents
        + "The topic is described by the following keywords: " + keywords + ".\n"
        + "Based on the documents and keywords, what is this topic about?"
    )

    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = t5_model.generate(**inputs, max_new_tokens=10)
    rows[topic]["Flan-T5"] = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

pd.DataFrame(rows)

**Interpret the output.**

1. For each keyword method, name one topic where it changed the c-TF-IDF keywords and say whether
   the change made the topic easier or harder to name.
2. Look at the Flan-T5 column. If two topics got the same label, say what that tells you about using
   a small generative model for this, and what you would change first: the model, the prompt, or
   the input documents.
3. None of the four descriptions moved a document. You report a topic's size and its name to
   someone who will not read the abstracts. Say which description you would send, and why.

**Your answer:**

_Type here._

---

## Optional extensions

The extensions need only the model fitted in Part 1, so you can run Parts 0 and 1 and jump straight
here. The figures come from the clusters, not the keywords, and Part 4 never moves a document, so
they look the same with or without it. Part 4 ends by resetting the keywords to c-TF-IDF, so any
keywords in these figures are the ones from Part 2.

> Library reference: [Visualize Topics](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html)
> in the BERTopic documentation shows each of these figures with its options.

### Optional Extension A · A word cloud for one topic

A word cloud sizes each keyword by its c-TF-IDF score. Ten keywords make a thin cloud, so we ask
`update_topics` for 100 keywords per topic first.

- **Python note:** `WordCloud.generate_from_frequencies` takes a dictionary from word to score. The
  loop builds that dictionary one keyword at a time.

In [ ]:
%pip install -q wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 100 keywords per topic instead of 10
topic_model.update_topics(abstracts, top_n_words=100)

topic = 0

frequencies = {}
for word, score in topic_model.get_topic(topic):
    frequencies[word] = score

cloud = WordCloud(background_color="white", width=1600, height=800)
cloud.generate_from_frequencies(frequencies)

plt.figure(figsize=(10, 5))
plt.imshow(cloud, interpolation="bilinear")
plt.axis("off")
plt.show()

### Optional Extension B · A topic similarity heatmap

BERTopic keeps one vector per topic, the average of its documents' embeddings, the same vectors
`find_topics` searched in Part 3. `visualize_heatmap` compares every pair of those vectors with
cosine similarity and draws the result as a matrix. Each row and each column is a topic, and a
darker cell means the two topics sit closer together.

- **Python note:** `top_n_topics=20` keeps the 20 largest topics so the matrix stays readable. The
  figure is interactive: hover over a cell to see the two topics and their similarity.

In [ ]:
# cosine similarity between the vectors of the 20 largest topics
topic_model.visualize_heatmap(top_n_topics=20)

> **Your turn.**
>
> Find the darkest cell off the diagonal. Print the keywords of those two topics with
> `topic_model.get_topic` and say whether they should be one topic or two.

In [ ]:
# Your turn: keywords of the two closest topics.

### Optional Extension C · A map of the topics

`visualize_topics` places every topic as a circle on a 2D map. Topics with similar keywords sit
close together, and a bigger circle means more abstracts. It is the heatmap from Extension B drawn
as a picture instead of a table.

- **Python note:** the map is interactive. Hover over a circle to see the topic's keywords, and use
  the slider underneath to highlight one topic at a time.

In [ ]:
# every topic as a circle, sized by its number of abstracts
topic_model.visualize_topics()

> **Your turn.**
>
> Find two circles that overlap. Hover over both and say what their keywords have in common.

**Your answer:**

_Type here._

### Optional Extension D · A tree of the topics

`visualize_hierarchy` joins the most similar topics step by step until one tree remains. The
closer to the left two branches join, the more alike the topics are. Cutting the tree further right
would merge topics into broader ones.

In [ ]:
# the 30 largest topics joined into a tree by similarity
topic_model.visualize_hierarchy(top_n_topics=30)

> **Your turn.**
>
> Find the first pair of topics to join. Say what broader topic they would form if merged.

**Your answer:**

_Type here._

### Optional Extension E · Topics over time

Each abstract has a publication year. `topics_over_time` counts how many abstracts each topic has
per year, and `visualize_topics_over_time` draws one line per topic.

- **Python note:** `topics_over_time` takes the abstracts and a list of years in the same order.

In [ ]:
years = list(dataset["Years"])

topics_over_time = topic_model.topics_over_time(abstracts, years)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=8)

> **Your turn.**
>
> Pick one topic whose line rises steeply in recent years. Say what change in NLP research could
> explain it.

**Your answer:**

_Type here._

### Optional Extension F · A map of the abstracts

Extension C drew one circle per topic. `visualize_documents` draws one dot per abstract, colored by
its topic, like the 2D plot in Part 1 of this lab but with BERTopic's labels. We pass the titles
instead of the abstracts, so hovering over a dot shows a short title rather than a whole abstract.

- **Python note:** the map needs two numbers per abstract. The 5 UMAP numbers BERTopic used are too
  many to draw, so we reduce the embeddings to 2 numbers with a second UMAP first.

> Library reference: [Visualize Documents](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#custom-hover)
> in the BERTopic documentation, the custom hover section.

In [ ]:
# 2 numbers per abstract, only for drawing
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric="cosine",
                          random_state=42).fit_transform(embeddings)

# one dot per abstract, colored by topic, with the title on hover
topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings)

> **Your turn.**
>
> Hover over a few grey dots, the outliers in topic -1. Say whether their titles look like they
> belong to a nearby topic or to no topic at all.

**Your answer:**

_Type here._

### Optional Extension G · How fast the keyword scores fall

BERTopic shows 10 keywords per topic by default. `visualize_term_rank` plots each keyword's c-TF-IDF
score against its rank, one line per topic: rank 1 is the top keyword, rank 2 the next, and so on.
Where a line flattens, the extra keywords add little. That bend, the elbow, is a way to choose how
many keywords a topic needs.

- **Python note:** we ask `update_topics` for 30 keywords per topic first so the lines have room to
  fall. `topics=[0, 1, 2, 3, 4, 5, 6, 7]` keeps the eight largest topics so the lines stay readable,
  and `log_scale=True` spreads out the small scores at the bottom.

> Library reference: [Visualize Terms](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_terms.html#visualize-term-score-decline)
> in the BERTopic documentation, the term score decline section.

In [ ]:
# 30 keywords per topic instead of 10
topic_model.update_topics(abstracts, top_n_words=30)

# c-TF-IDF score against keyword rank for the eight largest topics
topic_model.visualize_term_rank(topics=[0, 1, 2, 3, 4, 5, 6, 7], log_scale=True)

> **Your turn.**
>
> Find the rank where most lines flatten. Say whether 10 keywords per topic is too many, too few or
> about right for this data.

**Your answer:**

_Type here._

---

## Check yourself

- Name the step BERTopic adds to the pipeline from Part 1 of this lab.
- Say what makes a word score high in c-TF-IDF.
- Say what `find_topics` compares the search phrase against.
- Say what a representation model changes and what it leaves alone.

## Summary

BERTopic runs embedding, UMAP and HDBSCAN, then describes each cluster with c-TF-IDF keywords: words
that are frequent in the cluster and rare in the others.

Topics are numbered by size, and topic `-1` holds the outliers.

`find_topics` compares a phrase with the average embedding of each topic, so it returns the closest
topics even when none of them is close.

Representation models change how a topic is described, never which documents are in it.
KeyBERTInspired and Maximal Marginal Relevance reorder or replace keywords, and a generative model
writes a label.

## References

- Alammar and Grootendorst, *Hands-On Large Language Models*, Chapter 5, Text Clustering and Topic
  Modeling. This notebook is adapted from the book's Chapter 5 notebook, run on a 5,000 abstract
  sample. The prompt in Part 4 is the book's.
- The book's Chapter 5 notebook:
  [on GitHub](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter05/Chapter%205%20-%20Text%20Clustering%20and%20Topic%20Modeling.ipynb)
- Grootendorst, [BERTopic: Neural topic modeling with a class-based TF-IDF procedure](https://arxiv.org/abs/2203.05794), 2022
- [BERTopic documentation](https://maartengr.github.io/BERTopic/index.html), including the
  [representation models page](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html)
- Dataset card: [maartengr/arxiv_nlp](https://huggingface.co/datasets/maartengr/arxiv_nlp)
- Model card: [thenlper/gte-small](https://huggingface.co/thenlper/gte-small)
- Model card: [google/flan-t5-small](https://huggingface.co/google/flan-t5-small)